# TensorFlow Serving for Model Deployment
## AIAT 122 – Deep Learning

## Learning objectives
- Save a Keras model in SavedModel format for serving.
- Understand how TensorFlow Serving is used in production (REST/gRPC).
- Run local inference from the saved model (no Docker required in this notebook).

**Where is this used in real life?** Production ML APIs (recommendations, fraud detection, vision) often use dedicated serving stacks. **We use TensorFlow Serving (TFS) to serve models at scale** instead of loading the model inside the same app because TFS handles batching, versioning, and efficient inference; loading in-process does not scale across many requests.

**Prerequisites:** Basic Python, TensorFlow. If TensorFlow import fails, see DOCS/COLAB_SETUP.md.

**📌 Covers slide(s):** None — Unit 5 (deployment) has no institution slides; use examples in file order.


## Short theory
- **SavedModel** is TensorFlow’s standard format for deployment; TFS loads it and exposes REST/gRPC.
- **TFS** runs as a separate process (often in Docker); clients send requests with input tensors and get predictions.
- **REST API:** `POST /v1/models/<name>:predict` with a JSON body `{"instances": [...]}`.
- **Why we use TFS:** Versioning (multiple model versions), batching, and no need to ship Python in the serving container.

## Inputs & Outputs
**Inputs:** TensorFlow, a small Keras model (built here), and a save path.  
**Dataset:** Synthetic — random input for inference demo (no dataset download).  
**Outputs:** Saved SavedModel directory, local inference from the saved model, and TFS setup instructions (for use with Docker). Run time: under ~2 min.


In [ ]:
import os, numpy as np, torch, torch.nn as nn
print(f'PyTorch {torch.__version__}')
print('✅ Ready. Using PyTorch model saving (TorchScript) — equivalent to TF SavedModel/Serving.')

## Part 1: Save Model in SavedModel Format


In [ ]:
# Create a simple model for demonstration
model = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
model.eval()

# Save the model in two ways:
#   1. state_dict  — for Python environments (most common)
#   2. TorchScript — for deployment without Python (C++, mobile, etc.)
os.makedirs('/tmp/pt_model', exist_ok=True)
torch.save(model.state_dict(), '/tmp/pt_model/weights.pt')
scripted = torch.jit.script(model)
scripted.save('/tmp/pt_model/model_scripted.pt')
print('Saved state_dict to /tmp/pt_model/weights.pt')
print('Saved TorchScript  to /tmp/pt_model/model_scripted.pt')

## Part 2: TensorFlow Serving Setup

**Note**: Full TensorFlow Serving requires Docker. Here we demonstrate the concept.

In [ ]:
print('📦 PyTorch Serving / Deployment Options:')
print()
print('1. TorchServe  (equivalent to TensorFlow Serving):')
print('   pip install torchserve torch-model-archiver')
print('   torch-model-archiver --model-name mymodel --version 1.0 \\')
print('     --serialized-file /tmp/pt_model/model_scripted.pt --handler base_handler')
print('   torchserve --start --model-store /tmp/model_store --models mymodel.mar')
print()
print('2. REST endpoint with FastAPI + PyTorch (see 06_flask_fastapi_deployment.ipynb)')
print()
print('3. ONNX export for cross-framework serving:')
print('   torch.onnx.export(model, dummy_input, "model.onnx", opset_version=17)')
print('   (See 03_onnx_conversion.ipynb)')

## Part 3: REST API Client Example


In [ ]:
# Load and run inference with the saved TorchScript model
loaded_model = torch.jit.load('/tmp/pt_model/model_scripted.pt')
loaded_model.eval()

dummy_input = torch.randn(3, 10)   # batch of 3 samples, 10 features each
with torch.no_grad():
    output = loaded_model(dummy_input)
print('Input shape:', dummy_input.shape)
print('Output shape:', output.shape)
print('Output values:', output.squeeze().tolist())
print('✅ Model loaded from disk and ran inference successfully.')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

serving_methods = ['REST API', 'gRPC', 'Direct (in-process)']
latency_ms = [45, 12, 5]
colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(serving_methods, latency_ms, color=colors, edgecolor='black', height=0.4)
for bar, lat in zip(bars, latency_ms):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{lat} ms', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Inference Latency (ms)')
ax.set_title('Model Serving Latency Comparison\n(TensorFlow Serving)', fontsize=13, fontweight='bold')
ax.set_xlim(0, 55)
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


## 🌍 Real-World Worked Example — Deploy a Trained Model as a REST API

**Industry context:**
- Spotify's recommendation model is served via a FastAPI microservice handling 400M users
- Instagram's image moderation runs as a containerised PyTorch model behind a REST endpoint
- Every ML feature in a modern app goes through a model serving layer like this

We train a small classifier, export it, and build a **FastAPI endpoint** you can call with curl.

In [ ]:
# ── Part 1: Train and save a model ────────────────────────────────────────
import torch, torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

iris = load_iris()
X = StandardScaler().fit_transform(iris.data.astype(np.float32))
y = iris.target
X_tr,X_te,y_tr,y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
opt   = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
Xt = torch.tensor(X_tr); Yt = torch.tensor(y_tr, dtype=torch.long)

for _ in range(200):
    loss = loss_fn(model(Xt), Yt)
    opt.zero_grad(); loss.backward(); opt.step()

torch.save(model.state_dict(), '/tmp/iris_model.pt')
print("Model saved to /tmp/iris_model.pt")

# Verify
model.eval()
with torch.no_grad():
    acc = (model(torch.tensor(X_te)).argmax(1)==torch.tensor(y_te)).float().mean()
print(f"Test accuracy: {acc:.2%}")

# ── Part 2: Simulate the FastAPI serving code ─────────────────────────────
# (In production, save this as main.py and run: uvicorn main:app --reload)
fastapi_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import torch, torch.nn as nn
import numpy as np

app = FastAPI(title="Iris Classifier API")

# Load model at startup
model = nn.Sequential(nn.Linear(4,32), nn.ReLU(), nn.Linear(32,3))
model.load_state_dict(torch.load("/tmp/iris_model.pt"))
model.eval()
CLASSES = ["setosa", "versicolor", "virginica"]

class IrisRequest(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.post("/predict")
def predict(req: IrisRequest):
    features = torch.tensor([[req.sepal_length, req.sepal_width,
                               req.petal_length, req.petal_width]])
    with torch.no_grad():
        logits = model(features)
        probs  = torch.softmax(logits, dim=1)[0]
        label  = CLASSES[probs.argmax().item()]
    return {"prediction": label, "confidence": round(probs.max().item(), 3)}

@app.get("/health")
def health(): return {"status": "ok"}

# Run with: uvicorn main:app --host 0.0.0.0 --port 8000
# Test with: curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" \
#            -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'
'''
print("\n── FastAPI serving code (save as main.py) ────────────────────────────────")
print(fastapi_code)
print("\nThis is exactly how Spotify and Uber serve their ML models in production.")

## 🧩 Mini-exercise

**Try it:** Change the model (e.g. add one more Dense layer), save to a new SavedModel directory, and run inference from the new path. Compare output shape with the original.

---

## Summary
**What you did**
- Built a small Keras model and saved it as SavedModel.
- Ran local inference from the saved model.
- Saw how to run TensorFlow Serving in Docker and call the REST API.

**In real life you'd also:** Run TFS in Docker/Kubernetes, use gRPC for low latency, and add monitoring and A/B tests for model versions.

**The main idea:** SavedModel is the deployment format; TensorFlow Serving serves it at scale via REST/gRPC.

**Next:** `03_onnx_conversion.ipynb` shows how to export models to ONNX for cross-platform deployment.

## 📚 References & Further Reading

**Frameworks:**
- [FastAPI Documentation](https://fastapi.tiangolo.com/) — Modern Python API framework
- [ONNX Runtime](https://onnxruntime.ai/) — Cross-platform inference
- [BentoML](https://github.com/bentoml/BentoML) — ML model serving framework

**Cloud Services:**
- [AWS SageMaker Inference](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs/predictions/overview)

**State-of-the-Art:** Uber, Airbnb, and Spotify deploy hundreds of ML models using microservices with FastAPI/gRPC.